# New-rotor sensitivity model (SAL-001 winged sail)

Independent implementation of the per-decay momentum + thermal-noise sensitivity
calculation. Physics follows the same approach as Dave's `momentum-simulation/`
scripts but is written from scratch here; his published numbers are used only as
cross-check targets (last section).

**Geometry:** Pb-212 implanted ~30 nm into Kapton tape, applied to one face of each
Al wing (SAL-001, Al 1100, 0.1 mm). Two wings on opposite faces -> torques add.
The Al wing is the absorber backing (stops all downward alphas).

**Per Pb-212 decay:** exactly one alpha + heavy daughter, back-to-back:
64.06% Po-212 (8.78 MeV alpha, Pb-208 daughter), 35.94% Bi-212 (6.05 MeV alpha, Tl-208 daughter).
Net sail momentum = alphas that escape up (recoil) + daughters that escape up when the
alpha goes down (their range ~70-90 nm vs ~30 nm implantation depth).

In [1]:
import numpy as np

# -- physics constants --
MEV_C = 5.344e-22          # 1 MeV/c in kg m/s
M_ALPHA_MEV = 3727.379     # alpha rest energy, MeV
KB = 1.380649e-23          # J/K
T_BATH = 300.0             # K
T_HALF = 10.64 * 3600      # Pb-212 half-life, s
LAM = np.log(2) / T_HALF   # Pb-212 decay constant, 1/s

# -- decay branches: (prob, E_alpha MeV, daughter range in Kapton m) --
BRANCHES = [(0.6406, 8.78, 90e-9),   # Po-212 alpha, Pb-208 daughter
            (0.3594, 6.05, 70e-9)]   # Bi-212 alpha, Tl-208 daughter
D_IMPLANT = 30e-9          # typical implantation depth (peak of distribution)

def p_alpha_MeVc(E_MeV):
    return np.sqrt(2 * M_ALPHA_MEV * E_MeV)

# -- NEW ROTOR (SAL-001 + slotted disk), blessed 2026-07-16 --
I_ROTOR = 1.7e-11          # kg m^2, SolidWorks spin-axis principal moment (0.00017 g cm^2)
R_WING_IN, R_WING_OUT = 0.88e-3, 1.88e-3   # coated panel span from spin axis, m
R_EFF = 0.5 * (R_WING_IN + R_WING_OUT)     # centroid of uniform full-wing coating
N_WINGS = 2                # opposite faces, torques add

# -- damping scenarios (1/s): to be replaced by measured gamma of the new rotor --
GAMMAS = {"placeholder (10 min, Dave's doc)": 1/600,
          "old-rotor best (post-tilt 7/2)": 2.4e-4}
print(f"r_eff = {R_EFF*1e3:.2f} mm, I = {I_ROTOR:.2e} kg m^2")

r_eff = 1.38 mm, I = 1.70e-11 kg m^2


## Per-decay momentum to the sail

Closed form at fixed depth d, isotropic emission (u = |cos theta|):
- alpha up (half of decays): escapes from ~30 nm with ~full momentum -> contributes p*u, mean p/4
- alpha down: absorbed in Kapton+Al; daughter goes up, escapes if d/u < R_d,
  exit momentum p*sqrt(1 - d/(R_d u)) (heavy-ion p ~ sqrt(E), E ~ range)

In [2]:
def pz_per_decay_MeVc(d=D_IMPLANT):
    """Branching-weighted <p_z> per Pb-212 decay (MeV/c), closed form at depth d."""
    total = 0.0
    for prob, E, R_d in BRANCHES:
        p = p_alpha_MeVc(E)
        u = np.linspace(1e-6, 1, 20000)
        term_alpha_up = 0.5 * p * np.trapz(u, u)                  # = p/4
        esc = 1 - d / (R_d * u)
        integrand = np.where(esc > 0, u * np.sqrt(np.clip(esc, 0, None)), 0.0)
        term_daughter = 0.5 * p * np.trapz(integrand, u)
        total += prob * (term_alpha_up + term_daughter)
    return total

pz_closed = pz_per_decay_MeVc()
print(f"<p_z> closed form (d = 30 nm): {pz_closed:.1f} MeV/c")
print(f"alpha-only (no daughters) would be: "
      f"{sum(pr * p_alpha_MeVc(E)/4 for pr, E, _ in BRANCHES):.1f} MeV/c")

<p_z> closed form (d = 30 nm): 94.9 MeV/c
alpha-only (no daughters) would be: 60.1 MeV/c


In [3]:
# Mini Monte Carlo (vectorized) with the triangular implantation-depth distribution
rng = np.random.default_rng(0)
N = 1_000_000
pick = rng.random(N) < BRANCHES[0][0]
p = np.where(pick, p_alpha_MeVc(BRANCHES[0][1]), p_alpha_MeVc(BRANCHES[1][1]))
R_d = np.where(pick, BRANCHES[0][2], BRANCHES[1][2])
d = rng.triangular(0, 30e-9, 60e-9, size=N)
c = rng.uniform(-1, 1, size=N)

pz_ev = np.zeros(N)
up = c < 0
pz_ev[up] = p[up] * (-c[up])                       # alpha escapes, sail recoils
down = ~up
esc = 1 - d[down] / (R_d[down] * c[down])          # daughter escape fraction^2
pz_ev[down] = np.where(esc > 0, p[down] * np.sqrt(np.clip(esc, 0, None)) * c[down], 0.0)

pz_mc = pz_ev.mean()
PZ_SI = pz_mc * MEV_C                              # kg m/s per decay = N per Bq
print(f"<p_z> MC (depth-averaged): {pz_mc:.1f} MeV/c")
print(f"Force per Bq on the sail:  {PZ_SI:.2e} N/Bq")

<p_z> MC (depth-averaged): 94.9 MeV/c
Force per Bq on the sail:  5.07e-20 N/Bq


## Torque, steady-state omega, thermal noise

Adiabatic steady state (gamma >> lambda): omega_ss = alpha_signal / gamma.
Thermal noise on time-averaged omega (Ornstein-Uhlenbeck, exact for any gamma*T):
sigma_avg^2 = 2 sigma_inst^2 (gT - 1 + e^-gT)/(gT)^2, with sigma_inst = sqrt(kB T / I).

In [4]:
def alpha_per_Bq_wing(pz_SI=None, r_eff=R_EFF, I=I_ROTOR):
    return N_WINGS * (pz_SI if pz_SI else PZ_SI) * r_eff / I

def sigma_omega_avg(T_obs, gamma, I=I_ROTOR):
    s2 = KB * T_BATH / I
    g = gamma * T_obs
    return np.sqrt(2 * s2 * (g - 1 + np.exp(-g)) / g**2)

def min_activity(T_obs, gamma, **kw):
    """Min Bq/wing for SNR=1 on omega_ss after averaging T_obs."""
    return sigma_omega_avg(T_obs, gamma, kw.get('I', I_ROTOR)) / (alpha_per_Bq_wing(**kw) / gamma)

print(f"alpha per Bq/wing = {alpha_per_Bq_wing():.2e} rad/s^2\n")
T_OBS = {"1 min": 60, "10 min": 600, "1 hr": 3600, "1 day": 86400, "5 days": 5*86400}
hdr = "T_obs      " + "".join(f"{k:>28}" for k in GAMMAS)
print(hdr + "\n" + "-" * len(hdr))
for name, t in T_OBS.items():
    row = f"{name:<11}"
    for g in GAMMAS.values():
        row += f"{min_activity(t, g):>24.0f} Bq/w"
    print(row)

print("\nSNR for full-wing coating scenarios (per wing):")
for A in (350, 1000, 20000):
    for gname, g in GAMMAS.items():
        snr_day = A / min_activity(86400, g)
        print(f"  A = {A:>6} Bq/wing, gamma = {gname:<32} SNR(1 day) = {snr_day:6.1f}")

# decay-weighted correction for long integrations (source dies as e^-lam t)
for t in (86400,):
    w = np.sqrt((1 - np.exp(-2*LAM*t)) / (2*LAM) / t)
    print(f"\n(decay weighting over {t/3600:.0f} h multiplies SNR by ~{w:.2f})")

alpha per Bq/wing = 8.23e-12 rad/s^2

T_obs      placeholder (10 min, Dave's doc)old-rotor best (post-tilt 7/2)
-------------------------------------------------------------------------
1 min                          3109 Bq/w                     454 Bq/w
10 min                         2711 Bq/w                     444 Bq/w
1 hr                           1666 Bq/w                     398 Bq/w
1 day                           371 Bq/w                     138 Bq/w
5 days                          166 Bq/w                      63 Bq/w

SNR for full-wing coating scenarios (per wing):
  A =    350 Bq/wing, gamma = placeholder (10 min, Dave's doc) SNR(1 day) =    0.9
  A =    350 Bq/wing, gamma = old-rotor best (post-tilt 7/2)   SNR(1 day) =    2.5
  A =   1000 Bq/wing, gamma = placeholder (10 min, Dave's doc) SNR(1 day) =    2.7
  A =   1000 Bq/wing, gamma = old-rotor best (post-tilt 7/2)   SNR(1 day) =    7.3
  A =  20000 Bq/wing, gamma = placeholder (10 min, Dave's doc) SNR(1 day) =   53.9


## Cross-check against Dave's published numbers

`momentum-simulation/README.md` quotes: `<p_z>` = 94.97 MeV/c (MC), 5.08e-20 N/Bq,
min A(1 hr) = 920 Bq/wing and min A(1 day) = 200 Bq/wing at
I = 1.88e-11, r_c = 2.625 mm, gamma = 1/(10 min). Same physics, his geometry:

In [5]:
I_D, RC_D, G_D = 1.88e-11, 2.625e-3, 1/600
kw = dict(r_eff=RC_D, I=I_D)
print(f"<p_z> mine = {pz_mc:.1f} MeV/c        (Dave MC: 94.97)")
print(f"N/Bq  mine = {PZ_SI:.2e}       (Dave: 5.08e-20)")
print(f"minA 1 hr  = {min_activity(3600,  G_D, **kw):.0f} Bq/wing      (Dave: 920)")
print(f"minA 1 day = {min_activity(86400, G_D, **kw):.0f} Bq/wing      (Dave: 200)")

<p_z> mine = 94.9 MeV/c        (Dave MC: 94.97)
N/Bq  mine = 5.07e-20       (Dave: 5.08e-20)
minA 1 hr  = 921 Bq/wing      (Dave: 920)
minA 1 day = 205 Bq/wing      (Dave: 200)
